In [125]:
import importlib
import sys
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / 'src/allocations.py').exists()), Path.cwd())
sys.path.insert(0, str(ROOT))
import src.allocations
import src.metrics2

importlib.reload(src.allocations)
importlib.reload(src.metrics2)
from tqdm.auto import tqdm

from src.metrics2 import cohort_metrics

In [ ]:
NEED_COL          = "gagne_sum_t"
EXP_NEED_COL      = "exp_need_{model}"
GROUP_COL         = "race"
METHOD            = "bootstrap"
MODEL_CLASS       = "nn_multiclass"
# MODEL_CLASS       = "nn_multiclass_finetune_full_76aad3c8aee4"
SAMPLE_SIZE       = 100
ALLOCATION_DIR    = ROOT / "data/results/allocations"

PRED_METHODS      = ["average"]
NEED_THRESHOLDS   = [1]
SELECTION_RATES   = [0.1]

In [ ]:
fp_raw = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
df_raw = pd.read_csv(fp_raw)

In [ ]:
cohort_metric_columns = [
    "pairwise_consistency",
    "systemic_exclusion_all",
    "systemic_exclusion_qualified",
]
prediction_cache = {}


def predictions_for(pred_method, need_threshold):
    key = (pred_method, need_threshold)
    if key not in prediction_cache:
        qualification_col = f"qualified_gagne_{need_threshold}"
        path = ROOT / f"data/predictions/test_{METHOD}_{MODEL_CLASS}_{pred_method}_{qualification_col}.csv"
        predictions = pd.read_csv(path)
        prediction_cache[key] = predictions.merge(
            df_raw[["person_id", GROUP_COL]], on="person_id", how="left", validate="many_to_one"
        )
    return prediction_cache[key]


def allocation_tag(pred_method, need_threshold, selection_rate):
    return (
        f"{METHOD}_{MODEL_CLASS}_{pred_method}_y-qualified_gagne_{need_threshold}"
        f"_rate-{selection_rate}_size-{SAMPLE_SIZE}"
    )


def read_allocations(path):
    return pd.read_csv(path, converters={"selected": literal_eval, "models": literal_eval})

def evaluate_allocations(df, df_people, algorithm_allocations, *, pred_method,
                         need_threshold, selection_rate):
    """Evaluate saved selections using true needs, with one row per model."""
    qualification_col = f"qualified_gagne_{need_threshold}"
    k = int(selection_rate * SAMPLE_SIZE)
    rows = []
    for _, cohort_info in df_people.iterrows():
        seed = cohort_info["seed"]
        iteration = cohort_info["iteration"]

        split_df = df.loc[(df["seed"] == seed) & (df["person_id"] >= 0)]
        split_df = split_df.set_index("person_id")
        cohort = split_df.loc[cohort_info["people"]]

        for algorithm, allocations in algorithm_allocations:
            cohort_allocations = allocations.loc[(allocations["seed"] == seed) & (allocations["iteration"] == iteration)]

            # metrics2 expects one selection per model, including repeated sets.
            selected_lists = [
                selected_ids
                for selected_ids, models in zip(cohort_allocations["selected"], cohort_allocations["models"])
                for _ in models
            ]
            if not selected_lists:
                continue
            cohort_stats = cohort_metrics(
                selected_lists,
                cohort.index.to_list(),
                cohort["y"].eq(1).to_numpy(),
            )
            additional_metrics = {name: cohort_stats[name] for name in cohort_metric_columns}

            # Count selected sets, not model rows or allocation_idx values.
            # Sorting makes [person_a, person_b] and [person_b, person_a] identical.
            unique_allocations = {
                tuple(sorted(selected_ids)) for selected_ids in cohort_allocations["selected"]
            }
            model_names = {
                model for models in cohort_allocations["models"] for model in models
            }
            num_unique_allocations = len(unique_allocations)
            num_models = len(model_names)
            unique_allocation_ratio = num_unique_allocations / num_models if num_models else np.nan

            for _, allocation in cohort_allocations.iterrows():
                selected = cohort.loc[allocation["selected"]]

                selected_0 = selected.loc[selected[GROUP_COL] == 0, NEED_COL]
                selected_1 = selected.loc[selected[GROUP_COL] == 1, NEED_COL]

                mean_0_tru = selected_0.mean()
                mean_1_tru = selected_1.mean()

                absolute_disparity_tru = abs(mean_1_tru - mean_0_tru)
                disparity_ratio_tru = np.nan
                if pd.notna(mean_0_tru) and pd.notna(mean_1_tru) and mean_0_tru != 0 and mean_1_tru != 0:
                    disparity_ratio_tru = max(mean_1_tru / mean_0_tru, mean_0_tru / mean_1_tru)

                for model in allocation["models"]:
                    rows.append({
                        "method": METHOD,
                        "algorithm": algorithm,
                        "qualification_column": qualification_col,
                        "seed": seed,
                        "iteration": iteration,
                        "model": model,
                        "k": k,
                        "k_0": len(selected_0),
                        "k_1": len(selected_1),
                        "n": SAMPLE_SIZE,
                        "tau": need_threshold,
                        "selection_rate": selection_rate,
                        "pred_variant": pred_method,
                        "tolerance_type": "add",
                        "expected_utility": selected[model].mean(),
                        "realized_utility": selected["y"].mean(),
                        "absolute_disparity": absolute_disparity_tru,
                        "disparity_ratio": disparity_ratio_tru,
                        "num_models": num_models,
                        "num_unique_allocations": num_unique_allocations,
                        "unique_allocation_ratio": unique_allocation_ratio,
                        **additional_metrics,
                    }) 

    return rows


In [ ]:
all_results = []
for PRED_METHOD in PRED_METHODS:
    for NEED_THRESHOLD in NEED_THRESHOLDS:
        df = predictions_for(PRED_METHOD, NEED_THRESHOLD)
        for SELECTION_RATE in tqdm(SELECTION_RATES):
            tag = allocation_tag(PRED_METHOD, NEED_THRESHOLD, SELECTION_RATE)
            df_people = pd.read_csv(ALLOCATION_DIR / f"people_{tag}.csv", converters={"people": literal_eval})
            for algorithm, prefix in [("OPT", "opt"), ("LDA-tru", "lda_tru"), ("LDA-exp", "lda_exp")]:
                allocations = read_allocations(ALLOCATION_DIR / f"{prefix}_{tag}.csv")
                all_results.extend(evaluate_allocations(
                    df, df_people, [(algorithm, allocations)],
                    pred_method=PRED_METHOD, need_threshold=NEED_THRESHOLD,
                    selection_rate=SELECTION_RATE,
                ))

df_results = pd.DataFrame(all_results)

100%|██████████| 1/1 [00:05<00:00,  5.67s/it]


In [130]:
df_results

,method,algorithm,qualification_column,seed,iteration,model,alpha,beta,k,k_0,...,expected_utility,realized_utility,absolute_disparity,disparity_ratio,num_models,num_unique_allocations,unique_allocation_ratio,pairwise_consistency,systemic_exclusion_all,systemic_exclusion_qualified
0,bootstrap,OPT,qualified_gagne_1,0,0,m_1,1,NaN,10,9,...,0.980856,1.0,1.777778,1.592593,100,70,0.700000,0.956667,0.80,0.629630
1,bootstrap,OPT,qualified_gagne_1,0,0,m_51,1,NaN,10,9,...,0.984786,1.0,1.777778,1.592593,100,70,0.700000,0.956667,0.80,0.629630
2,bootstrap,OPT,qualified_gagne_1,0,0,m_2,1,NaN,10,10,...,0.980218,1.0,NaN,NaN,100,70,0.700000,0.956667,0.80,0.629630
3,bootstrap,OPT,qualified_gagne_1,0,0,m_3,1,NaN,10,8,...,0.988039,1.0,0.375000,1.083333,100,70,0.700000,0.956667,0.80,0.629630
4,bootstrap,OPT,qualified_gagne_1,0,0,m_9,1,NaN,10,8,...,0.978500,1.0,0.375000,1.083333,100,70,0.700000,0.956667,0.80,0.629630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7465,bootstrap,LDA-exp,qualified_gagne_1,4,4,m_96,0,NaN,10,4,...,0.988772,1.0,0.583333,1.122807,98,96,0.979592,0.921847,0.76,0.630769
7466,bootstrap,LDA-exp,qualified_gagne_1,4,4,m_97,0,NaN,10,5,...,0.983204,1.0,1.400000,1.350000,98,96,0.979592,0.921847,0.76,0.630769
7467,bootstrap,LDA-exp,qualified_gagne_1,4,4,m_98,0,NaN,10,4,...,0.988464,1.0,1.750000,1.368421,98,96,0.979592,0.921847,0.76,0.630769
7468,bootstrap,LDA-exp,qualified_gagne_1,4,4,m_99,0,NaN,10,4,...,0.991730,1.0,2.083333,1.438596,98,96,0.979592,0.921847,0.76,0.630769


In [131]:
columns = ["absolute_disparity", "disparity_ratio", "k_0", "k_1"]
summary_keys = ["model", "selection_rate", "k", "n", "pred_variant", "algorithm"]
df_results[df_results["algorithm"]=="LDA-exp"].groupby(by=summary_keys, dropna=False)[columns].agg(["mean", "std"])[["absolute_disparity"]]

absolute_disparity  \
                                                                 mean   
model selection_rate k  n   pred_variant algorithm                      
m_1   0.1            10 100 average      LDA-exp             1.817045   
m_10  0.1            10 100 average      LDA-exp             1.697511   
m_100 0.1            10 100 average      LDA-exp             1.759217   
m_11  0.1            10 100 average      LDA-exp             1.422980   
m_12  0.1            10 100 average      LDA-exp             1.177116   
...                                                               ...   
m_95  0.1            10 100 average      LDA-exp             1.560317   
m_96  0.1            10 100 average      LDA-exp             1.275661   
m_97  0.1            10 100 average      LDA-exp             1.649892   
m_98  0.1            10 100 average      LDA-exp             1.422727   
m_99  0.1            10 100 average      LDA-exp             1.743561   

                                                              
                                                         std  
model selection_rate k  n   pred_variant algorithm            
m_1   0.1            10 100 average      LDA-exp    1.459927  
m_10  0.1            10 100 average      LDA-exp    1.164739  
m_100 0.1            10 100 average      LDA-exp    1.343577  
m_11  0.1            10 100 average      LDA-exp    0.871250  
m_12  0.1            10 100 average      LDA-exp    0.856616  
...                                                      ...  
m_95  0.1            10 100 average      LDA-exp    1.202279  
m_96  0.1            10 100 average      LDA-exp    0.993541  
m_97  0.1            10 100 average      LDA-exp    0.991515  
m_98  0.1            10 100 average      LDA-exp    0.952675  
m_99  0.1            10 100 average      LDA-exp    1.045857  

[100 rows x 2 columns]

In [132]:
columns = ["absolute_disparity", "disparity_ratio", "k_0", "k_1"]
summary_keys = ["seed", "selection_rate", "k", "n", "pred_variant", "algorithm"]
df_results[df_results["algorithm"]=="LDA-exp"].groupby(by=summary_keys, dropna=False)[columns].agg(["mean", "std"])[["absolute_disparity"]]

absolute_disparity          
                                                                mean       std
seed selection_rate k  n   pred_variant algorithm                             
0    0.1            10 100 average      LDA-exp             1.625762  1.086330
1    0.1            10 100 average      LDA-exp             1.955808  1.191571
2    0.1            10 100 average      LDA-exp             1.875013  1.332491
3    0.1            10 100 average      LDA-exp             1.081398  0.666127
4    0.1            10 100 average      LDA-exp             1.326749  0.916194

In [133]:
df_results[df_results["algorithm"]=="LDA-exp"][["k_0", "k_1", "absolute_disparity"]].describe()

,k_0,k_1,absolute_disparity
count,2490.000000,2490.000000,2172.000000
mean,4.031727,5.968273,1.547152
std,2.882352,2.882352,1.103950
min,0.000000,0.000000,0.000000
25%,1.000000,4.000000,0.666667
50%,4.000000,6.000000,1.333333
75%,6.000000,9.000000,2.222222
max,10.000000,10.000000,6.500000


In [134]:
columns = ["realized_utility", "expected_utility", "absolute_disparity", "disparity_ratio", "k_0", "k_1"]
summary_keys = ["selection_rate", "k", "n", "pred_variant", "algorithm"]
grouped = df_results.groupby(summary_keys, dropna=False)[columns].agg(["mean", "std"])

df_cohort_metrics = df_results.drop_duplicates(summary_keys + ["seed", "iteration"])
cohort_grouped = df_cohort_metrics.groupby(summary_keys, dropna=False)[cohort_metric_columns].agg(["mean", "std"])
grouped = pd.concat([grouped, cohort_grouped], axis=1)
columns = columns + cohort_metric_columns
summary = pd.DataFrame(index=grouped.index)

for col in columns:
    summary[col] = grouped.apply(lambda row: f"{row[(col, 'mean')]:.2f} \u00b1 {row[(col, 'std')]:.2f}", axis=1)
summary

realized_utility  \
selection_rate k  n   pred_variant algorithm                    
0.1            10 100 average      LDA-exp        0.98 ± 0.04   
                                   LDA-tru        0.98 ± 0.05   
                                   OPT            0.98 ± 0.04   

                                             expected_utility  \
selection_rate k  n   pred_variant algorithm                    
0.1            10 100 average      LDA-exp        0.99 ± 0.00   
                                   LDA-tru        0.99 ± 0.00   
                                   OPT            0.99 ± 0.00   

                                             absolute_disparity  \
selection_rate k  n   pred_variant algorithm                      
0.1            10 100 average      LDA-exp          1.55 ± 1.10   
                                   LDA-tru          0.06 ± 0.25   
                                   OPT              1.97 ± 1.41   

                                             disparity_ratio          k_0  \
selection_rate k  n   pred_variant algorithm                                
0.1            10 100 average      LDA-exp       1.60 ± 0.73  4.03 ± 2.88   
                                   LDA-tru       1.03 ± 0.15  4.14 ± 3.09   
                                   OPT           1.65 ± 0.71  4.11 ± 3.10   

                                                      k_1  \
selection_rate k  n   pred_variant algorithm                
0.1            10 100 average      LDA-exp    5.97 ± 2.88   
                                   LDA-tru    5.86 ± 3.09   
                                   OPT        5.89 ± 3.10   

                                             pairwise_consistency  \
selection_rate k  n   pred_variant algorithm                        
0.1            10 100 average      LDA-exp            0.93 ± 0.01   
                                   LDA-tru            0.93 ± 0.01   
                                   OPT                0.94 ± 0.01   

                                             systemic_exclusion_all  \
selection_rate k  n   pred_variant algorithm                          
0.1            10 100 average      LDA-exp              0.74 ± 0.04   
                                   LDA-tru              0.74 ± 0.04   
                                   OPT                  0.77 ± 0.03   

                                             systemic_exclusion_qualified  
selection_rate k  n   pred_variant algorithm                               
0.1            10 100 average      LDA-exp                    0.59 ± 0.06  
                                   LDA-tru                    0.59 ± 0.06  
                                   OPT                        0.64 ± 0.05